## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <cstdio>
using namespace std;

struct Move {
    int type, val;
    Move(int t, int v) : type(t), val(v) {}
};

struct PermutationRestorer {
    int n, a, b, l;
    vector<int> board;
    vector<int> pos;
    vector<Move> moves;

    PermutationRestorer(int size, int val_a, int val_b, const vector<int>& initial_board) {
        n = size;
        a = val_a;
        b = val_b;
        board = initial_board;
        pos.assign(n, 0);
        moves.clear();

        int diff = (a - b + n) % n;
        l = diff & -diff;
        if (l == 0) l = n;

        for (int i = 0; i < n; ++i)
            pos[board[i]] = i;
    }

    void apply_add(int x) {
        if (x == 0) return;
        moves.emplace_back(2, x);
        for (int i = 0; i < n; ++i) {
            board[i] = (board[i] + x) % n;
            pos[board[i]] = i;
        }
    }

    void apply_xor(int x) {
        if (x == 0) return;
        moves.emplace_back(1, x);
        for (int i = 0; i < n; ++i) {
            board[i] ^= x;
            pos[board[i]] = i;
        }
    }

    void append_add(int x) {
        if (x != 0) moves.emplace_back(2, x);
    }

    void append_xor(int x) {
        if (x != 0) moves.emplace_back(1, x);
    }

    vector<int> compute_target_positions(int u, int v) {
        int delta = (v - u + 2 * n - l) % n;
        int pu = 0, pv = 0;
        int step = n / 2;
        while (step >= 2 * l) {
            if (delta >= step) {
                delta -= step;
                pv += step / 2;
            } else {
                pu += step / 2;
            }
            step /= 2;
        }
        pu += n / 2;
        pu += u & (l - 1);
        pv += u & (l - 1);
        return {pu, pv};
    }

    void swap_two_values(int c, int d) {
        if (c == d) return;
        int block_c = (c / l) % 2;
        int block_d = (d / l) % 2;
        if (block_c == block_d) {
            int pivot = (block_c == 0) ? ((c & (l - 1)) + l) : (c & (l - 1));
            swap_two_values(c, pivot);
            swap_two_values(d, pivot);
            swap_two_values(c, pivot);
            return;
        }

        vector<int> p_ab = compute_target_positions(a, b);
        vector<int> p_cd = compute_target_positions(c, d);
        int pa = p_ab[0], pb = p_ab[1];
        int pc = p_cd[0], pd = p_cd[1];

        append_add((pc - c + n) % n);
        append_xor(pc ^ pa);
        append_add((a - pa + n) % n);
        moves.emplace_back(0, 0);
        append_add((pa - a + n) % n);
        append_xor(pc ^ pa);
        append_add((c - pc + n) % n);

        int pc_idx = pos[c];
        int pd_idx = pos[d];
        swap(board[pc_idx], board[pd_idx]);
        pos[c] = pd_idx;
        pos[d] = pc_idx;
    }

    bool resolve_low_order(const vector<int>& sequence, int m, vector<int>& out_ops) {
        vector<bool> seen(m, false);
        for (int x : sequence) {
            if (x < 0 || x >= m || seen[x])
                return false;
            seen[x] = true;
        }
        if (m == 1) return true;

        int half = m / 2;
        vector<int> even_part(half), odd_part(half);
        for (int i = 0; i < half; ++i) {
            even_part[i] = sequence[2 * i] / 2;
            odd_part[i] = sequence[2 * i + 1] / 2;
        }

        vector<int> even_ops, odd_ops;
        if (!resolve_low_order(even_part, half, even_ops)) return false;
        if (!resolve_low_order(odd_part, half, odd_ops)) return false;

        vector<int> combined_ops;
        if (sequence[0] % 2 != 0) {
            combined_ops.push_back(m == 2 ? 1 : -1);
        }

        int tb = 0;
        for (int op : even_ops) {
            if (op > 0) {
                combined_ops.push_back(-1);
                combined_ops.push_back(1);
            } else {
                combined_ops.push_back(op * 2);
                tb ^= (-op * 2);
            }
        }
        if (tb != 0) combined_ops.push_back(-tb);

        int tc = 0;
        for (int op : odd_ops) {
            if (op > 0) {
                combined_ops.push_back(1);
                combined_ops.push_back(-1);
            } else {
                combined_ops.push_back(op * 2);
                tc ^= (-op * 2);
            }
        }

        if ((tc & half) != (tb & half)) {
            for (int i = 0; i < half / 2; ++i) {
                combined_ops.push_back(-1);
                combined_ops.push_back(1);
            }
        }
        if (tb >= half) tb -= half;
        if (tc >= half) tc -= half;
        if (tb != tc) return false;

        for (int op : combined_ops) {
            if (out_ops.empty()) {
                out_ops.push_back(op);
            } else if (op < 0 && out_ops.back() < 0) {
                int last = out_ops.back();
                out_ops.pop_back();
                int new_xor = -((-last) ^ (-op));
                if (new_xor != 0)
                    out_ops.push_back(new_xor);
            } else {
                out_ops.push_back(op);
            }
        }
        return true;
    }

    bool restore() {
        if (l > 1) {
            vector<int> low_bits(l);
            for (int i = 0; i < l; ++i)
                low_bits[i] = board[i] & (l - 1);
            vector<int> ops;
            if (!resolve_low_order(low_bits, l, ops))
                return false;
            for (int op : ops) {
                if (op > 0) apply_add(op);
                else apply_xor(-op);
            }
        }

        for (int rem = 0; rem < l; ++rem) {
            vector<int> actual;
            for (int idx = rem; idx < n; idx += l)
                actual.push_back(board[idx]);
            sort(actual.begin(), actual.end());
            int expected = rem;
            for (int x : actual) {
                if (x != expected)
                    return false;
                expected += l;
            }
        }

        for (int rem = 0; rem < l; ++rem) {
            for (int idx = rem; idx < n; idx += l) {
                while (board[idx] != idx) {
                    swap_two_values(idx, board[idx]);
                }
            }
        }

        for (int i = 0; i < n; ++i) {
            if (board[i] != i)
                return false;
        }
        return true;
    }

    void printSolution() const {
        printf("%d\n", (int)moves.size());
        for (const Move& op : moves) {
            if (op.type == 0) {
                printf("0\n");
            } else {
                printf("%d %d\n", op.type, op.val);
            }
        }
    }
};

int main() {
    int n;
    if (scanf("%d", &n) != 1) return 0;
    int a, b;
    scanf("%d %d", &a, &b);
    vector<int> init(n);
    for (int i = 0; i < n; ++i)
        scanf("%d", &init[i]);

    PermutationRestorer solver(n, a, b, init);
    if (!solver.restore()) {
        printf("-1\n");
    } else {
        solver.printSolution();
    }
    return 0;
}

## B 长跑

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

struct Station {
    int p; 
    int c; 
    bool operator<(const Station& other) const {
        return p < other.p;
    }
};

void solve() {
    int N, L, Maxn, S;
    while (cin >> N >> L >> Maxn >> S) {
        vector<Station> stations(N);
        for (int i = 0; i < N; ++i) {
            cin >> stations[i].p >> stations[i].c;
        }
        if (L <= Maxn) {
            cout << "Yes\n";
            continue;
        }

        sort(stations.begin(), stations.end());
        
        const long long INF = 1e18;
        vector<long long> dp(N, INF);
        
        for (int i = 0; i < N; ++i) {
            if (stations[i].p <= Maxn) {
                dp[i] = stations[i].c;
            }
            for (int j = 0; j < i; ++j) {
                if (stations[i].p - stations[j].p <= Maxn) {
                    if (dp[j] != INF) {
                        dp[i] = min(dp[i], dp[j] + stations[i].c);
                    }
                }
            }
        }
        
        long long min_cost = INF;
        for (int i = 0; i < N; ++i) {
            if (L - stations[i].p <= Maxn) {
                min_cost = min(min_cost, dp[i]);
            }
        }
        if (min_cost <= S) {
            cout << "Yes\n";
        } else {
            cout << "No\n";
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}

## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ull = unsigned long long;

const ull BASE = 131;

// Manacher 算法：预处理字符串，返回插入#后的字符串，并计算每个位置的回文半径
string manacher(const string& s, vector<int>& p) {
    string t = "$#";
    for (char c : s) {
        t += c;
        t += '#';
    }
    int m = t.size();
    p.assign(m, 0);
    int center = 0, right = 0;
    for (int i = 1; i < m; i++) {
        if (i < right) {
            p[i] = min(p[2 * center - i], right - i);
        } else {
            p[i] = 1;
        }
        while (i - p[i] >= 0 && i + p[i] < m && t[i - p[i]] == t[i + p[i]]) {
            p[i]++;
        }
        if (i + p[i] > right) {
            right = i + p[i];
            center = i;
        }
    }
    return t;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;
    string A, B;
    cin >> A >> B;

    vector<int> pa, pb;
    string ta = manacher(A, pa);
    string tb = manacher(B, pb);

    int m = ta.size();
    vector<ull> powBase(m + 1, 1);
    for (int i = 1; i <= m; i++) {
        powBase[i] = powBase[i - 1] * BASE;
    }

    // 预处理 A 的正向哈希
    vector<ull> preA(m + 1, 0);
    for (int i = 0; i < m; i++) {
        preA[i + 1] = preA[i] * BASE + ta[i];
    }

    // 预处理 B 的反向哈希（相当于把 B 逆序后做前缀哈希）
    vector<ull> sufB(m + 2, 0);
    for (int i = m - 1; i >= 0; i--) {
        sufB[i] = sufB[i + 1] * BASE + tb[i];
    }

    // 获取 A 中 [l, r] 的哈希
    auto getAHash = [&](int l, int r) -> ull {
        if (l > r) return 0;
        return preA[r + 1] - preA[l] * powBase[r - l + 1];
    };

    // 获取 B 中 [l, r] 的反向哈希（等价于 B 逆序后 [l, r] 的哈希）
    auto getBReverseHash = [&](int l, int r) -> ull {
        if (l > r) return 0;
        return sufB[l] - sufB[r + 1] * powBase[r - l + 1];
    };

    int ans = 0;

    // 遍历所有可能的拼接中心
    for (int i = 2; i < m; i++) {
        int j = i - 2; // B 中对应的中心位置
        if (j < 0 || j >= m) continue;

        // 取基础回文长度
        int len = max(pa[i], pb[j]);
        ans = max(ans, len - 1); // 去掉 # 后的真实长度

        // 计算基础回文在 ta、tb 中的边界
        int leftA = i - len + 1;
        int rightB = j + len - 1;

        // 二分查找最多能扩展多少字符
        int l = 0, r = min(leftA, m - 1 - rightB);
        int add = 0;
        while (l <= r) {
            int mid = (l + r) / 2;
            int aL = leftA - mid;
            int aR = leftA - 1;
            int bL = rightB + 1;
            int bR = rightB + mid;

            if (getAHash(aL, aR) == getBReverseHash(bL, bR)) {
                add = mid;
                l = mid + 1;
            } else {
                r = mid - 1;
            }
        }

        ans = max(ans, len - 1 + add);
    }

    cout << ans << '\n';
    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <vector>
#include <set>

using namespace std;

const int MAX_X = 100005; 
int status_x[MAX_X];      
int last_I[MAX_X];       
int last_O[MAX_X];       

void solve() {
    int m;
    while (cin >> m) {
        if (m == 0) {
            cout << -1 << "\n";
            continue;
        }

        set<int> available_q; 
        vector<int> seen_x; 
        
        int ans = -1;
        bool error_found = false;

        for (int i = 1; i <= m; ++i) {
            char op;
            cin >> op;
            if (op == '?') {
                if (!error_found) {
                    available_q.insert(i);
                }
            } else {
                int x;
                cin >> x;
                
                if (error_found) continue;

                seen_x.push_back(x);

                if (op == 'I') {
                    if (status_x[x] == 1) {
                        int limit = last_I[x];
                        auto it = available_q.upper_bound(limit);
                        if (it != available_q.end()) {
                            last_O[x] = *it; 
                            available_q.erase(it);
                            
                            last_I[x] = i;  
                        } else {
                            ans = i;
                            error_found = true;
                        }
                    } else {
                        status_x[x] = 1;
                        last_I[x] = i;
                    }
                } else if (op == 'O') {
                    if (status_x[x] == 0) {
                        int limit = last_O[x];
                        auto it = available_q.upper_bound(limit);
                        if (it != available_q.end()) {
                            last_I[x] = *it; 
                            available_q.erase(it);
                            
                            last_O[x] = i;   
                        } else {
                            ans = i;
                            error_found = true;
                        }
                    } else {
                        status_x[x] = 0;
                        last_O[x] = i;
                    }
                }
            }
        }
        
        cout << ans << "\n";
        
        for (int x : seen_x) {
            status_x[x] = 0;
            last_I[x] = 0;
            last_O[x] = 0;
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    solve();
    return 0;
}

## E 任意点

In [ ]:
#include <iostream>
#include <vector>

using namespace std;

struct Point {
    int x, y;
};

class DSU {
public:
    vector<int> parent;
    int components; 

    DSU(int n) {
        parent.resize(n);
        for (int i = 0; i < n; ++i) {
            parent[i] = i; 
        }
        components = n;   
    }

    int find(int i) {
        if (parent[i] == i) {
            return i;
        }
        return parent[i] = find(parent[i]);
    }

    void unite(int i, int j) {
        int root_i = find(i);
        int root_j = find(j);
        if (root_i != root_j) {
            parent[root_i] = root_j;
            components--; 
        }
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    while (cin >> n) {
        vector<Point> points(n);
        for (int i = 0; i < n; ++i) {
            cin >> points[i].x >> points[i].y;
        }

        DSU dsu(n);

        for (int i = 0; i < n; ++i) {
            for (int j = i + 1; j < n; ++j) {
                if (points[i].x == points[j].x || points[i].y == points[j].y) {
                    dsu.unite(i, j);
                }
            }
        }

        cout << dsu.components - 1 << "\n";
    }

    return 0;
}

## F 通配符匹配

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <string_view>
#include <algorithm>

using namespace std;

// 存储被 '*' 切割后的每一个片段的信息
struct Chunk {
    int length;                                   // 该片段的总长度（包含 '?'）
    vector<pair<int, string_view>> exact_parts;   // 内部被 '?' 切割出的纯字母段及其相对偏移量
    int anchor_offset;                            // 最长纯字母段（锚点）的偏移量
    string_view anchor;                           // 最长纯字母段，用于极速 find 定位
};

// 解析片段，提取出最长纯字母段作为查找锚点
Chunk parse_chunk(string_view p) {
    Chunk chunk;
    chunk.length = p.length();
    chunk.anchor_offset = 0;
    
    int offset = 0;
    int start = 0;
    for (int i = 0; i <= p.length(); ++i) {
        if (i == p.length() || p[i] == '?') {
            if (i > start) {
                chunk.exact_parts.push_back({start, p.substr(start, i - start)});
            }
            start = i + 1;
        }
    }
    
    // 寻找最长的子串作为锚点
    int max_len = -1;
    for (const auto& part : chunk.exact_parts) {
        if ((int)part.second.length() > max_len) {
            max_len = part.second.length();
            chunk.anchor_offset = part.first;
            chunk.anchor = part.second;
        }
    }
    
    return chunk;
}

// 严丝合缝校验某个确定的位置是否与 Chunk 匹配
bool match_chunk(string_view s, int start_idx, const Chunk& chunk) {
    for (const auto& part : chunk.exact_parts) {
        if (s.substr(start_idx + part.first, part.second.length()) != part.second) {
            return false;
        }
    }
    return true;
}

// 在 s[start_idx : end_limit] 区间内贪心查找最早匹配的位置
int find_chunk(string_view s, int start_idx, int end_limit, const Chunk& chunk) {
    if (chunk.anchor.empty()) {
        // 如果全是由 '?' 组成
        if (start_idx + chunk.length <= end_limit) return start_idx;
        return -1;
    }
    
    int search_start = start_idx + chunk.anchor_offset;
    // 保证整个 chunk 长度不会越界
    int search_end_limit = end_limit - chunk.length + chunk.anchor_offset + chunk.anchor.length();
    
    while (true) {
        size_t idx = s.find(chunk.anchor, search_start);
        if (idx == string_view::npos || (int)idx > search_end_limit) {
            return -1;
        }
        
        int cand_start = (int)idx - chunk.anchor_offset;
        
        bool match = true;
        for (const auto& part : chunk.exact_parts) {
            if (part.first == chunk.anchor_offset) continue; // 锚点本身不需要重复比较
            if (s.substr(cand_start + part.first, part.second.length()) != part.second) {
                match = false;
                break;
            }
        }
        
        if (match) {
            return cand_start;
        }
        search_start = idx + 1; // 找下一个可能的位置
    }
}

int main() {
    // 极致解除流同步，加速 I/O 读写
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    string pattern;
    if (!(cin >> pattern)) return 0;
    
    int n;
    cin >> n;
    
    vector<string> files(n);
    for (int i = 0; i < n; ++i) {
        cin >> files[i];
    }
    
    vector<string_view> parts;
    int start = 0;
    for (int i = 0; i <= pattern.length(); ++i) {
        if (i == pattern.length() || pattern[i] == '*') {
            parts.push_back(string_view(pattern).substr(start, i - start));
            start = i + 1;
        }
    }
    
    // 情况一：完全没有 '*'，纯粹精准匹配或问号匹配
    if (parts.size() == 1) {
        Chunk chunk = parse_chunk(parts[0]);
        for (const string& f : files) {
            if ((int)f.length() == chunk.length && match_chunk(f, 0, chunk)) {
                cout << "YES\n";
            } else {
                cout << "NO\n";
            }
        }
        return 0;
    }
    
    // 情况二：存在 '*'
    Chunk pref_chunk = parse_chunk(parts.front());
    Chunk suff_chunk = parse_chunk(parts.back());
    
    vector<Chunk> mid_chunks;
    for (int i = 1; i < (int)parts.size() - 1; ++i) {
        if (!parts[i].empty()) {  // 忽略连续 '*' 产生的空段
            mid_chunks.push_back(parse_chunk(parts[i]));
        }
    }
    
    int min_required_len = pref_chunk.length + suff_chunk.length;
    for (const auto& c : mid_chunks) {
        min_required_len += c.length;
    }
    
    for (const string& f : files) {
        if ((int)f.length() < min_required_len) {
            cout << "NO\n";
            continue;
        }
        
        // 1. 强匹配前缀
        if (pref_chunk.length > 0 && !match_chunk(f, 0, pref_chunk)) {
            cout << "NO\n";
            continue;
        }
        
        // 2. 强匹配后缀
        if (suff_chunk.length > 0 && !match_chunk(f, f.length() - suff_chunk.length, suff_chunk)) {
            cout << "NO\n";
            continue;
        }
        
        // 3. 贪心搜索中间的区块
        int pos = pref_chunk.length;
        int end_limit = f.length() - suff_chunk.length;
        bool possible = true;
        
        for (const auto& chunk : mid_chunks) {
            int match_pos = find_chunk(f, pos, end_limit, chunk);
            if (match_pos == -1) {
                possible = false;
                break;
            }
          
            pos = match_pos + chunk.length;
        }
        
        if (possible) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }
    
    return 0;
}

## G 汉诺塔

In [ ]:
#include <iostream>
#include <vector>
#include <string>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<string> prior(6);
    for (int i = 0; i < 6; ++i) {
        cin >> prior[i];
    }
    
    vector<vector<long long>> dp(n + 1, vector<long long>(3, 0));
    vector<vector<int>> dest(n + 1, vector<int>(3, 0));

    for (int i = 0; i < 3; ++i) {
        char from = 'A' + i;
        for (int j = 0; j < 6; ++j) {
            if (prior[j][0] == from) {
                dest[1][i] = prior[j][1] - 'A';
                dp[1][i] = 1;
                break; 
            }
        }
    }
    for (int i = 2; i <= n; ++i) {
        for (int j = 0; j < 3; ++j) {
            int pos_i = j;     
            int pos_small = j;  
            long long steps = 0;
            
            while (true) {
                int next_small = dest[i - 1][pos_small];
                steps += dp[i - 1][pos_small];
                pos_small = next_small;
                
                if (pos_small == pos_i) {
                    break;
                }
                int other = 3 - pos_i - pos_small; 
                steps += 1; 
                pos_i = other;
            }
            
            dp[i][j] = steps;
            dest[i][j] = pos_i;
        }
    }

    cout << dp[n][0] << "\n";

    return 0;
}

## H 马步距离

In [ ]:
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    long long xp, yp, xs, ys;
    while (cin >> xp >> yp >> xs >> ys) {
        long long dx = abs(xp - xs);
        long long dy = abs(yp - ys);

        if (dx < dy) {
            swap(dx, dy);
        }

        if (dx == 1 && dy == 0) {
            cout << 3 << "\n";
            continue;
        }
        if (dx == 2 && dy == 2) {
            cout << 4 << "\n";
            continue;
        }
        long long ans = max((dx + 1) / 2, (dx + dy + 2) / 3);

        if ((ans % 2) != ((dx + dy) % 2)) {
            ans++;
        }
        cout << ans << "\n";
    }

    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        heights.push_back(0);
        stack<int> st;
        int max_area = 0;
        
        for (int i = 0; i < heights.size(); ++i) {
            while (!st.empty() && heights[i] < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();
                int w = st.empty() ? i : i - st.top() - 1;
                
                max_area = max(max_area, h * w);
            }
            st.push(i);
        }
        
        return max_area;
    }
};

## J 消防局的设立

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <queue>

using namespace std;

vector<vector<int>> tree;
vector<int> depth;
vector<int> parent_node;
vector<bool> covered;
vector<int> sorted_nodes;

void dfs(int u, int p, int d) {
    depth[u] = d;
    parent_node[u] = p;
    sorted_nodes.push_back(u);
    for (int v : tree[u]) {
        if (v != p) {
            dfs(v, u, d + 1);
        }
    }
}

// 标记以 start 为中心，距离不超过 2 的所有点
void mark(int start) {
    queue<pair<int, int>> q;
    q.push({start, 0});
    covered[start] = true;
    // 记录在这次 mark 中访问过哪些节点，防止距离计算出错，或者直接用 covered 就行，但被其他消防局覆盖过的点也应该继续穿透，所以最好传 parent
}

void mark_dfs(int u, int p, int dist) {
    covered[u] = true;
    if (dist == 0) return;
    for (int v : tree[u]) {
        if (v != p) {
            mark_dfs(v, u, dist - 1);
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    tree.resize(n + 1);
    depth.resize(n + 1, 0);
    parent_node.resize(n + 1, 0);
    covered.resize(n + 1, false);

    for (int i = 2; i <= n; ++i) {
        int a;
        cin >> a;
        tree[i].push_back(a);
        tree[a].push_back(i);
    }

    dfs(1, 0, 1);

    sort(sorted_nodes.begin(), sorted_nodes.end(), [](int a, int b) {
        return depth[a] > depth[b];
    });

    int ans = 0;
    for (int u : sorted_nodes) {
        if (!covered[u]) {
            int p = parent_node[u];
            int gp = p == 0 ? 0 : parent_node[p];
            
            int center = gp;
            if (center == 0) {
                center = (p == 0) ? u : p;
            }

            ans++;
            // 标记以 center 为中心，距离不超过 2 的点
            mark_dfs(center, 0, 2);
        }
    }

    cout << ans << "\n";
    return 0;
}